<a id="graph"></a>

$$
\text{图论模型 (Graph Theory): 图 } G = (V, E) \text{, V 是顶点 (Vertex / Node), E 是边 (Edge)} \\
\text{带权图: 每条边 (i, j) 有一个权重 } w_{ij} \text{ (距离, 时间, 成本)} \\ \quad \\
\text{邻接矩阵 (Adjacency Matrix): n 个顶点的 } n \times n \text{ 矩阵} \\
W_{ij} = \begin{cases} w_{ij} & (i, j) \in E \\ 0 & i = j \\ \infty & \text{没有边} \end{cases} \\
\text{无向图 (Undirected): W 是对称矩阵; 有向图 (Directed): 不一定对称} \\
\text{邻接表 (Adjacency List): 每个顶点只存它的邻居, 稀疏图更省空间} \\ \quad \\
\text{这一节的问题:} \\
\text{1. 单源最短路径 (Single-Source Shortest Path): 从一个起点到所有点 → Dijkstra} \\
\text{2. 所有点对最短路径 (All-Pairs Shortest Path) → Floyd-Warshall} \\
\text{3. 最小生成树 (Minimum Spanning Tree): 用总权重最小的边把所有点连起来 → Prim / Kruskal}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import shortest_path, minimum_spanning_tree

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

names = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
pos = np.array([[0, 1], [1, 2], [1, 0], [2, 1], [3, 2], [3, 0], [4, 1], [5, 1]], dtype=float)
edges = [(0, 1, 4), (0, 2, 2), (1, 2, 5), (1, 3, 10), (2, 3, 3), (1, 4, 7), (3, 4, 4),
         (3, 5, 6), (2, 5, 12), (4, 6, 3), (5, 6, 2), (4, 5, 8), (6, 7, 5), (4, 7, 11)]
n = len(names)
W = np.full((n, n), np.inf)
np.fill_diagonal(W, 0)
for i, j, w in edges:
    W[i, j] = W[j, i] = w     # 无向图
print(f'邻接矩阵 W = \n{W}')
print(f'对称? {np.allclose(np.where(np.isinf(W), -1, W), np.where(np.isinf(W.T), -1, W.T))}')

def draw_graph(highlight=(), title=''):
    plt.figure(figsize=(4, 3))
    hl = {tuple(sorted(e)) for e in highlight}
    for i, j, w in edges:
        on = tuple(sorted((i, j))) in hl
        plt.plot(*pos[[i, j]].T, color='r' if on else 'gray', linewidth=2.5 if on else 1, zorder=1)
        plt.text(*(pos[i] + pos[j]) / 2, str(w), fontsize=8, color='b', ha='center', va='center',
                 bbox=dict(facecolor='white', edgecolor='none', pad=0.5))
    plt.scatter(*pos.T, s=250, c='#cce5ff', edgecolors='k', zorder=2)
    for k, name in enumerate(names):
        plt.text(*pos[k], name, ha='center', va='center', zorder=3)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

draw_graph(title='weighted undirected graph')

<a id="dijkstra"></a>

$$
\text{Dijkstra 算法 (要求所有边权} \geq 0 \text{)} \\ \quad \\
d[v] \text{: 目前找到的从起点 s 到 v 的最短距离 (临时值), 初始 } d[s] = 0 \text{, 其余 } \infty \\
S \text{: 已经确定最短距离的顶点集合, 初始为空} \\
\text{重复 n 次:} \\
\text{1. 在不属于 S 的顶点中选 d 最小的 u, 加入 S} \\
\text{2. 松弛 (Relaxation): 对 u 的每个邻居 v, 若 } d[u] + w_{uv} < d[v] \text{, 更新 } d[v] = d[u] + w_{uv} \text{, 记录前驱 } prev[v] = u \\ \quad \\
\text{正确性 (贪心不变量): u 被选出时, d[u] 一定是真正的最短距离} \\
\text{任取一条 s 到 u 的路径 P. s 在 S 里, u 不在, 所以 P 上一定有第一条离开 S 的边 } (x, y) \text{, } x \in S, y \notin S \\
\text{x 已确定, 加入 S 时松弛过 } (x, y) \text{, 所以 } d[y] \leq d[x] + w_{xy} \leq \text{P 走到 y 的长度} \\
\text{u 是 S 外 d 最小的, 所以 } d[u] \leq d[y] \\
\text{P 从 y 到 u 剩下的边权都} \geq 0 \text{, 所以 P 的长度} \geq \text{P 走到 y 的长度} \geq d[y] \geq d[u] \\
\text{任何路径都不比 d[u] 短} \implies d[u] \text{ 是最短距离} \\ \quad \\
\text{路径重建: 从终点沿 prev 往回走到起点, 再反转} \\
\text{(最短路径的子路径也是最短路径, 这是最优性原理, 所以存一个前驱就够了)} \\
\text{复杂度: 用数组 } O(n^2) \text{, 用堆 (优先队列) } O((n + |E|) \log n)
$$

对比: [动态规划的最优性原理](dynamic_programming.ipynb#dp-overview)

In [ ]:
def dijkstra(W, s):
    n = len(W)
    d = np.full(n, np.inf); d[s] = 0
    prev = np.full(n, -1)
    in_S = np.zeros(n, dtype=bool)
    order = []
    for _ in range(n):
        u = int(np.argmin(np.where(in_S, np.inf, d)))
        if np.isinf(d[u]):
            break                      # 剩下的点不连通
        in_S[u] = True
        order.append(u)
        for v in range(n):
            if not in_S[v] and np.isfinite(W[u, v]) and d[u] + W[u, v] < d[v]:
                d[v] = d[u] + W[u, v]
                prev[v] = u
    return d, prev, order

def rebuild_path(prev, t):
    path = []
    while t != -1:
        path.append(int(t))
        t = prev[t]
    return path[::-1]

d, prev, order = dijkstra(W, 0)
print(f'确定顺序: {[names[k] for k in order]}')
for v in range(n):
    print(f'A -> {names[v]}: 距离 = {d[v]:.0f}, 路径 = {"-".join(names[k] for k in rebuild_path(prev, v))}')

d_scipy = shortest_path(np.where(np.isinf(W), 0, W), method='D', directed=False, indices=0)
print(f'和 scipy (Dijkstra) 一致? {np.allclose(d, d_scipy)}')

path_AH = rebuild_path(prev, 7)
draw_graph(highlight=list(zip(path_AH[:-1], path_AH[1:])), title=f'shortest path A -> H, length {d[7]:.0f}')

In [ ]:
# 负权边时贪心不变量失效: 证明中 "剩下的边权 >= 0" 这一步不成立
W_neg = np.array([[0, 2, 5],
                  [np.inf, 0, np.inf],
                  [np.inf, -4, 0]], dtype=float)     # 有向图: s->a 2, s->b 5, b->a -4
d_neg, _, _ = dijkstra(W_neg, 0)
d_bf = shortest_path(np.where(np.isinf(W_neg), 0, W_neg), method='BF', directed=True, indices=0)
print(f'有负权边: Dijkstra 得到 d = {d_neg}, Bellman-Ford 得到 d = {d_bf}')
print('Dijkstra 先确定了 a (距离 2), 但 s -> b -> a = 5 - 4 = 1 更短')

<a id="floyd"></a>

$$
\text{Floyd-Warshall 算法: 所有点对的最短路径 (动态规划)} \\ \quad \\
\text{状态: } d_k(i, j) = \text{从 i 到 j, 中间只允许经过顶点 } \{1, 2, \cdots, k\} \text{ 的最短距离} \\
d_0(i, j) = W_{ij} \text{ (不经过任何中间点, 只能直接走一条边)} \\ \quad \\
\text{推导递推式: 考虑允许经过 } \{1, \cdots, k\} \text{ 的最短路径, 只有两种情况:} \\
\text{1. 不经过 k: 那它只用了 } \{1, \cdots, k - 1\} \implies d_{k-1}(i, j) \\
\text{2. 经过 k: (没有负环时) k 只出现一次, 路径分成 } i \to k \text{ 和 } k \to j \text{ 两段, 两段都只用 } \{1, \cdots, k - 1\} \text{, 都要最短} \\
\quad \implies d_{k-1}(i, k) + d_{k-1}(k, j) \\
d_k(i, j) = \min\left( d_{k-1}(i, j), \ d_{k-1}(i, k) + d_{k-1}(k, j) \right) \\
d_n(i, j) \text{ 就是答案 (允许经过所有顶点)} \\ \quad \\
\text{实现: 只需要一个矩阵原地更新 (第 k 轮里 } d(i, k), d(k, j) \text{ 不会变, 因为 } d(k, k) = 0 \text{)} \\
\text{用 numpy 广播: } D \leftarrow \min(D, \ D_{:, k} + D_{k, :}) \text{, 复杂度 } O(n^3) \\
\text{路径重建: 记录 } nxt(i, j) \text{ = 从 i 去 j 的第一步; 经过 k 更短时 } nxt(i, j) \leftarrow nxt(i, k)
$$

In [ ]:
def floyd_warshall(W):
    n = len(W)
    D = W.copy()
    nxt = np.where(np.isfinite(W), np.arange(n)[None, :], -1)    # 直接走一条边时, 第一步就是终点
    for k in range(n):
        via_k = D[:, k:k + 1] + D[k:k + 1, :]
        better = via_k < D
        D = np.where(better, via_k, D)
        nxt = np.where(better, nxt[:, k:k + 1], nxt)
    return D, nxt

def fw_path(nxt, i, j):
    if nxt[i, j] == -1:
        return []
    path = [i]
    while i != j:
        i = int(nxt[i, j])
        path.append(i)
    return path

D, nxt = floyd_warshall(W)
print(f'所有点对最短距离 = \n{D}')
D_scipy = shortest_path(np.where(np.isinf(W), 0, W), method='FW', directed=False)
print(f'和 scipy (Floyd-Warshall) 一致? {np.allclose(D, D_scipy)}')
print(f'第 0 行和 Dijkstra 一致? {np.allclose(D[0], d)}')
print(f'B -> G: {"-".join(names[k] for k in fw_path(nxt, 1, 6))}, 距离 = {D[1, 6]:.0f}')

# 更大的随机有向图测试
rng = np.random.default_rng(5)
n_big = 60
W_big = np.where(rng.random((n_big, n_big)) < 0.1, rng.uniform(1, 10, (n_big, n_big)), np.inf)
np.fill_diagonal(W_big, 0)
D_big, _ = floyd_warshall(W_big)
D_big_scipy = shortest_path(np.where(np.isinf(W_big), 0, W_big), method='D', directed=True)
dij_rows = np.array([dijkstra(W_big, s)[0] for s in range(n_big)])
print(f'60 个点的随机有向图: Floyd == scipy? {np.allclose(D_big, D_big_scipy)}, Dijkstra (每个起点各跑一次) == Floyd? {np.allclose(dij_rows, D_big)}')

<a id="mst"></a>

$$
\text{最小生成树 (Minimum Spanning Tree, MST)} \\
\text{生成树: 包含所有 n 个顶点, 没有环, 连通的子图, 一定恰好有 n - 1 条边} \\
\text{(连通至少要 n - 1 条边; 树多一条边就会出现环)} \\
\text{最小生成树: 边权之和最小的生成树 (例: 用最少的电缆把所有城市连起来)} \\ \quad \\
\text{切割性质 (Cut Property): 把顶点分成两部分 S 和 V - S, 横跨两部分的边中权重最小的那条 e, 一定属于某棵最小生成树} \\
\text{证明 (交换论证): 设最小生成树 T 不包含 e = (u, v), 把 e 加进 T 会形成一个环} \\
\text{这个环从 S 走到 V - S (通过 e) 又必须走回 S, 所以环上还有另一条横跨的边 e'} \\
\text{去掉 e' 得到 } T' = T + e - e' \text{, 仍然连通, n - 1 条边, 是生成树} \\
w(T') = w(T) + w(e) - w(e') \leq w(T) \text{ (因为 e 是横跨边中最小的)} \implies T' \text{ 也是最小生成树, 且包含 e} \\ \quad \\
\text{Prim 算法: 从一个顶点出发, S = 已经连上的顶点, 每次加入横跨 S 和 V - S 的最小边 (直接用切割性质)} \\
\text{Kruskal 算法: 把所有边按权重从小到大排序, 依次加入, 如果会形成环就跳过} \\
\text{(加入边 (u, v) 时, 取 S = u 所在的连通分量, (u, v) 是横跨它的最小边, 所以也由切割性质保证正确)} \\
\text{判断 "会不会形成环" = u 和 v 是否已经在同一个连通分量: 用并查集 (Union-Find)}
$$

In [ ]:
def prim(W):
    n = len(W)
    in_tree = np.zeros(n, dtype=bool); in_tree[0] = True
    tree = []
    for _ in range(n - 1):
        # 横跨 S 和 V - S 的所有边里选最小的
        cross = np.where(in_tree[:, None] & ~in_tree[None, :], W, np.inf)
        i, j = np.unravel_index(np.argmin(cross), cross.shape)
        tree.append((int(i), int(j), W[i, j]))
        in_tree[j] = True
    return tree

def kruskal(n, edges):
    parent = list(range(n))
    def find(x):                      # 并查集: 找根, 顺便压缩路径
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    tree = []
    for i, j, w in sorted(edges, key=lambda e: e[2]):
        ri, rj = find(i), find(j)
        if ri != rj:                  # 不在同一个连通分量, 加入不会形成环
            parent[ri] = rj
            tree.append((i, j, w))
    return tree

tree_prim = prim(W)
tree_kruskal = kruskal(n, edges)
mst_scipy = minimum_spanning_tree(np.where(np.isinf(W), 0, W))
print(f'Prim:    {[f"{names[i]}{names[j]}({w:.0f})" for i, j, w in tree_prim]}, 总权重 = {sum(w for *_, w in tree_prim):.0f}')
print(f'Kruskal: {[f"{names[i]}{names[j]}({w})" for i, j, w in tree_kruskal]}, 总权重 = {sum(w for *_, w in tree_kruskal)}')
print(f'scipy minimum_spanning_tree 总权重 = {mst_scipy.sum():.0f}, 边数 = {mst_scipy.nnz} (= n - 1 = {n - 1})')

# 随机完全图测试
rng = np.random.default_rng(6)
all_ok = True
for _ in range(20):
    m = int(rng.integers(5, 30))
    pts = rng.random((m, 2))
    Wr = np.linalg.norm(pts[:, None] - pts[None, :], axis=2)
    er = [(i, j, Wr[i, j]) for i in range(m) for j in range(i + 1, m)]
    t1 = sum(w for *_, w in prim(np.where(np.eye(m, dtype=bool), 0, Wr)))
    t2 = sum(w for *_, w in kruskal(m, er))
    t3 = minimum_spanning_tree(Wr).sum()
    all_ok &= np.isclose(t1, t3) and np.isclose(t2, t3)
print(f'20 个随机完全图: Prim, Kruskal, scipy 的总权重全部一致? {all_ok}')

draw_graph(highlight=[(i, j) for i, j, _ in tree_kruskal], title=f'minimum spanning tree, total {sum(w for *_, w in tree_kruskal)}')